# BioPython Essentials

BioPython is the most widely used Python library for bioinformatics. This notebook covers its core modules through practical, hands-on examples -- from sequence manipulation to fetching data from NCBI and building a complete gene-to-protein analysis workflow.

## Learning Objectives

- Create and manipulate `Seq` objects (complement, reverse complement, translate, transcribe)
- Work with `SeqRecord` objects (annotations, features, letter_annotations)
- Read and write biological file formats with `SeqIO` (FASTA, GenBank, FASTQ)
- Search and fetch data from NCBI using `Bio.Entrez`
- Parse sequence alignments with `AlignIO`
- Build a complete workflow: from fetching a gene to analyzing its protein product

**Prerequisites:** Basic Python, understanding of DNA/RNA/protein central dogma

## Why this notebook matters

BioPython is the foundation library for computational biology in Python. Almost every task in this course — reading sequences, downloading from databases, performing alignments, parsing annotation files — uses BioPython. This notebook covers the essential modules: `Bio.Seq` (sequence objects), `Bio.SeqRecord` (sequences with metadata), `Bio.SeqIO` (reading/writing FASTA, GenBank, FASTQ), `Bio.Entrez` (NCBI database access), and `Bio.Align` (pairwise alignment). By the end you will be able to fetch a gene from NCBI, extract its CDS, translate it, and analyze the protein.

## How to use this notebook

1. Run cells top-to-bottom in order — later cells depend on earlier ones
2. Run the environment check cell first to confirm all imports work
3. Read the explanatory text before each code cell — the context matters
4. The exercises at the end are designed to be done after reading each section
5. If a code cell requires internet access (Entrez, PDB download), it is marked — these can be skipped if offline

## Complicated moments explained

- **Seq vs. str**: A `Seq` object behaves like a Python string (supports slicing, len(), iteration) but adds biological methods: `.complement()`, `.reverse_complement()`, `.transcribe()`, `.translate()`. In modern BioPython (>=1.79), `Seq` and `str` can be freely combined in most contexts. Pre-1.79, you had to call `str()` explicitly.
- **translate() and stop codons**: `dna.translate()` translates all codons, representing stop codons as `*`. `dna.translate(to_stop=True)` stops at the first stop codon and omits the `*`. For extracting the protein product of a CDS, always use `to_stop=True`.
- **SeqIO.parse() vs SeqIO.read()**: `parse()` returns an *iterator* of SeqRecord objects — use it for multi-record files (always safe). `read()` returns a single SeqRecord — use it only when the file is guaranteed to contain exactly one record (raises an error otherwise).
- **Entrez rate limits**: Always call `handle.close()` after reading to release the connection. NCBI allows 3 requests/second without an API key. For batch work, get a free NCBI API key and set `Entrez.api_key = "yourkey"`.

## Environment check (run this first)

In [ ]:
# Environment check
import Bio
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
from Bio import SeqIO, Entrez, AlignIO
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.SeqUtils import gc_fraction, molecular_weight

print(f"BioPython version: {Bio.__version__}")

# Quick sanity checks
dna = Seq("ATGAAACCCGGGTAA")
protein = dna.translate(to_stop=True)
print(f"Seq object: {dna} -> translate(to_stop=True) -> {protein}")
print(f"GC content: {gc_fraction(dna)*100:.1f}%")

blosum62 = substitution_matrices.load("BLOSUM62")
print(f"BLOSUM62 loaded: score(L,I)={blosum62['L','I']}, score(K,D)={blosum62['K','D']}")
print("\nAll imports ready. Proceed to Section 1.")

In [ ]:
dna = Seq("ATGCGATCGATCGTAA")

print(f"Original:           5'-{dna}-3'")
print(f"Complement:         3'-{dna.complement()}-5'")
print(f"Reverse complement: 5'-{dna.reverse_complement()}-3'")
print()

# Transcription: DNA -> RNA (replaces T with U)
rna = dna.transcribe()
print(f"DNA: {dna}")
print(f"RNA: {rna}")

# Back-transcription: RNA -> DNA
print(f"Back to DNA: {rna.back_transcribe()}")
print(f"Round-trip OK: {rna.back_transcribe() == dna}")

### 1.2 Translation: DNA/RNA to Protein

Each triplet of nucleotides (codon) codes for one amino acid. The `*` character represents a stop codon.

In [ ]:
coding_dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

# Full translation (includes stop codon as *)
print(f"Full:        {coding_dna.translate()}")

# Stop at first stop codon (most common usage)
print(f"to_stop:     {coding_dna.translate(to_stop=True)}")
print()

# Different genetic codes
dna2 = Seq("ATGTGAATGGAA")
print(f"Codons:         ATG TGA ATG GAA")
print(f"Standard (1):   {dna2.translate(table=1)}  (TGA = stop)")
print(f"Mito (2):       {dna2.translate(table=2)}  (TGA = Trp)")
print(f"Bacterial (11): {dna2.translate(table=11)}  (TGA = stop)")

In [ ]:
# GC content and molecular weight
dna = Seq("ATGCGATCGATCGATCG")
print(f"GC content: {gc_fraction(dna) * 100:.1f}%")
print(f"DNA MW: {molecular_weight(dna):.1f} Da")

protein = Seq("MKPG")
print(f"Protein MW: {molecular_weight(protein, seq_type='protein'):.1f} Da")

# Mutable sequences (for in-place editing)
mutable = MutableSeq("ATGCGATCG")
mutable[3] = "T"  # G -> T point mutation
print(f"\nMutated: {mutable} (position 3: G->T)")

---
## 2. SeqRecord Objects

`SeqRecord` wraps a `Seq` object and adds metadata: ID, name, description, annotations, features, and per-letter annotations. This is the main data structure you get when reading files with `SeqIO`.

In [ ]:
# Creating a SeqRecord with annotations
record = SeqRecord(
    Seq("ATGCGATCGATCGATCGATCGATCGTAA"),
    id="BRCA1_001",
    name="BRCA1",
    description="Breast cancer type 1 susceptibility protein, partial CDS"
)

# Record-level annotations
record.annotations["molecule_type"] = "DNA"
record.annotations["organism"] = "Homo sapiens"
record.annotations["taxonomy"] = ["Eukaryota", "Metazoa", "Chordata", "Mammalia"]

print(f"ID: {record.id}  |  Name: {record.name}")
print(f"Description: {record.description}")
print(f"Sequence: {record.seq}  ({len(record)} bp)")
print(f"Annotations: {dict(record.annotations)}")

In [ ]:
# Adding features (annotated regions on the sequence)
gene_feature = SeqFeature(
    FeatureLocation(start=0, end=28), type="gene",
    qualifiers={"gene": ["BRCA1"]}
)
cds_feature = SeqFeature(
    FeatureLocation(start=0, end=27), type="CDS",
    qualifiers={"gene": ["BRCA1"], "product": ["BRCA1 protein"]}
)
record.features = [gene_feature, cds_feature]

for feat in record.features:
    print(f"{feat.type:6s} {feat.location}  qualifiers={dict(feat.qualifiers)}")

# Extract the nucleotide sequence of a feature
cds_seq = cds_feature.location.extract(record.seq)
print(f"\nCDS protein: {cds_seq.translate(to_stop=True)}")

In [ ]:
# Per-letter annotations (e.g., quality scores in FASTQ)
short_record = SeqRecord(Seq("ATGCGATCG"), id="read001")
short_record.letter_annotations["phred_quality"] = [30, 30, 28, 35, 35, 33, 30, 25, 20]

for base, qual in zip(short_record.seq, short_record.letter_annotations["phred_quality"]):
    print(f"  {base} Q={qual}", end="")
print(f"\nMean quality: {sum(short_record.letter_annotations['phred_quality']) / len(short_record):.1f}")

---
## 3. SeqIO: Reading and Writing Sequence Files

`SeqIO` provides a uniform interface for many biological file formats:
- `SeqIO.parse(file, fmt)` -- read multiple records (iterator)
- `SeqIO.read(file, fmt)` -- read exactly one record
- `SeqIO.write(records, file, fmt)` -- write records
- `SeqIO.to_dict(...)` -- load into a dictionary keyed by ID
- `SeqIO.convert(in_file, in_fmt, out_file, out_fmt)` -- format conversion

### 3.1 FASTA Files

In [ ]:
# Create a sample FASTA file
fasta_content = """>INS_HUMAN Insulin [Homo sapiens]
MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKT
RREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN
>INS_MOUSE Insulin [Mus musculus]
MALWTRLLPLLALLALWAPAPAHAFVNQHLCGSHLVEALYLVCGERGFFYTPKS
RREVEDPQVAQLELGGGPGADDLQTLALEVAQQKRGIVDQCCTSICSLYQLENYCN
>INS_RAT Insulin [Rattus norvegicus]
MALWMHLLTVLALLALWGPNSVQAFVKQHLCGSHLVEALYLVCGERGFFYTPKS
RREVEDPQVEQLELGGSPGDLQTLALEVARQKRGIVDQCCTSICSLYQLENYCN
"""
with open("insulin_seqs.fasta", "w") as f:
    f.write(fasta_content)

# Read and display
for record in SeqIO.parse("insulin_seqs.fasta", "fasta"):
    print(f"{record.id:15s} {len(record):4d} aa  {record.description}")

In [ ]:
# Load into dictionary, write new file
records_dict = SeqIO.to_dict(SeqIO.parse("insulin_seqs.fasta", "fasta"))
print(f"Loaded IDs: {list(records_dict.keys())}")
print(f"Human insulin: {records_dict['INS_HUMAN'].seq[:40]}...")

# Write selected records
new_records = [
    SeqRecord(Seq("ATGCGATCGATCG"), id="seq1", description="Test sequence 1"),
    SeqRecord(Seq("GCTAGCTAGCTAG"), id="seq2", description="Test sequence 2"),
]
count = SeqIO.write(new_records, "output.fasta", "fasta")
print(f"\nWrote {count} sequences to output.fasta")

### 3.2 GenBank Files

In [ ]:
# Create a sample GenBank file
genbank_content = """LOCUS       EXAMPLE_GENE          120 bp    DNA     linear   PRI 01-JAN-2024
DEFINITION  Homo sapiens example gene, complete CDS.
ACCESSION   EX000001
VERSION     EX000001.1
KEYWORDS    .
SOURCE      Homo sapiens (human)
  ORGANISM  Homo sapiens
            Eukaryota; Metazoa; Chordata; Craniata; Vertebrata;
            Mammalia; Primates; Haplorrhini; Catarrhini; Hominidae;
            Homo.
FEATURES             Location/Qualifiers
     source          1..120
                     /organism="Homo sapiens"
                     /mol_type="mRNA"
     gene            1..120
                     /gene="EXAMPLE"
     CDS             11..100
                     /gene="EXAMPLE"
                     /codon_start=1
                     /product="example protein"
                     /protein_id="EXP00001.1"
                     /translation="MAISRVTLGERILKLCHELY"
     5'UTR           1..10
                     /gene="EXAMPLE"
     3'UTR           101..120
                     /gene="EXAMPLE"
ORIGIN
        1 cttcacagcg atggctatca gccgtgtgac actgggcgag cgtatcctga agctgtgcca
       61 tgagctctat tagcgatcga tcgatcgatc gatcgatcga tcgatcgatc gatcgatcga
//
"""
with open("example.gb", "w") as f:
    f.write(genbank_content)

# Read and explore
record = SeqIO.read("example.gb", "genbank")
print(f"ID: {record.id}  Name: {record.name}")
print(f"Description: {record.description}")
print(f"Length: {len(record)} bp")
print(f"Organism: {record.annotations.get('organism', 'N/A')}")

print(f"\nFeatures ({len(record.features)}):")
for feat in record.features:
    gene = feat.qualifiers.get('gene', [''])[0]
    print(f"  {feat.type:10s} {str(feat.location):20s} {gene}")

In [ ]:
# Extract CDS and translate
for feat in record.features:
    if feat.type == "CDS":
        cds_seq = feat.location.extract(record.seq)
        annotated = feat.qualifiers.get("translation", [""])[0]
        our_translation = cds_seq.translate(to_stop=True)
        
        print(f"Gene: {feat.qualifiers.get('gene', ['?'])[0]}")
        print(f"CDS location: {feat.location}")
        print(f"Annotated protein: {annotated}")
        print(f"Our translation:   {our_translation}")
        break

### 3.3 FASTQ Files

In [ ]:
# Create sample FASTQ, read, and filter by quality
fastq_content = """@read001 Example read 1
ATGCGATCGATCGATCGATCGATCGATCGATCGATCG
+
IIIIIIIHHHHHGGGGFFFFEEEEDDDDDCCCCBBBB
@read002 Example read 2
GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAG
+
IIIIIIIIIHHHHHHHGGGGGFFFFFEEEEEDDDDDCC
@read003 Low quality read
TTAACCGGTTAACCGGTTAACCGGTTAACCGGTTAAC
+
BBBBAAA@@@@?????>>>>>>======<<<<<<;;;;;
"""
with open("reads.fastq", "w") as f:
    f.write(fastq_content)

# Read and show quality stats
for record in SeqIO.parse("reads.fastq", "fastq"):
    quals = record.letter_annotations["phred_quality"]
    mean_q = sum(quals) / len(quals)
    print(f"{record.id:10s} len={len(record):3d}  mean_Q={mean_q:.1f}  min_Q={min(quals)}")

# Quality filtering
good = [r for r in SeqIO.parse("reads.fastq", "fastq")
        if sum(r.letter_annotations["phred_quality"]) / len(r) >= 25]
SeqIO.write(good, "filtered.fastq", "fastq")
print(f"\nKept {len(good)}/3 reads with mean Q >= 25")

In [ ]:
# Format conversion
# GenBank -> FASTA
records = list(SeqIO.parse("example.gb", "genbank"))
SeqIO.write(records, "from_genbank.fasta", "fasta")

# FASTQ -> FASTA (one-liner, note: quality info is lost)
count = SeqIO.convert("reads.fastq", "fastq", "reads.fasta", "fasta")
print(f"Converted {count} FASTQ records to FASTA")

with open("reads.fasta") as f:
    print(f.read())

---
## 4. Bio.Entrez: Accessing NCBI Databases

The `Entrez` module provides programmatic access to all NCBI databases. The key functions:
- `Entrez.esearch(db, term)` -- search, returns IDs
- `Entrez.efetch(db, id, rettype, retmode)` -- download records
- `Entrez.elink(dbfrom, db, id)` -- find cross-database links

In [ ]:
Entrez.email = "your.email@example.com"  # NCBI requires this

# Search for a gene
handle = Entrez.esearch(
    db="nucleotide",
    term="hemoglobin beta[Gene] AND Homo sapiens[Organism] AND RefSeq[Filter] AND mRNA[Filter]",
    retmax=5
)
results = Entrez.read(handle)
handle.close()

print(f"Total matches: {results['Count']}")
print(f"Returned IDs: {results['IdList']}")

In [ ]:
# Fetch a GenBank record and explore it
handle = Entrez.efetch(db="nucleotide", id="NM_000518.5", rettype="gb", retmode="text")
hbb_record = SeqIO.read(handle, "genbank")
handle.close()

print(f"Accession: {hbb_record.id}")
print(f"Description: {hbb_record.description}")
print(f"Length: {len(hbb_record)} bp")
print(f"Organism: {hbb_record.annotations.get('organism', 'N/A')}")
print(f"\nFeatures ({len(hbb_record.features)}):")
for feat in hbb_record.features:
    print(f"  {feat.type:12s} {feat.location}")

In [ ]:
# Fetch multiple sequences at once and save locally
accessions = ["NM_000518.5", "NM_000207.3", "NM_000546.6"]  # HBB, INS, TP53

handle = Entrez.efetch(db="nucleotide", id=",".join(accessions), rettype="fasta", retmode="text")
records = list(SeqIO.parse(handle, "fasta"))
handle.close()

print(f"Fetched {len(records)} sequences:")
for rec in records:
    print(f"  {rec.id}: {len(rec)} bp")

SeqIO.write(records, "ncbi_genes.fasta", "fasta")
print(f"\nSaved to ncbi_genes.fasta")

---
## 5. AlignIO: Sequence Alignments

In [ ]:
# Create sample alignment
alignment_content = """>HBB_HUMAN
MVHLTPEEKSAVTALWGKVN-VDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLG
>HBB_MOUSE
MVHLTDAEKAAVNGLWGKVN-ADEVGGEALGRLLVVYPWTQRFFDSFGDLSSASAIMGNPKVKAHGKKVIT
>HBB_CHICKEN
MVHWTAEEKQLITGLWGKVN-VAECGAEALARLLIVYPWTQRFFASFGNLSSPTAILGNPMVRAHGKKVLT
>HBB_FROG
MVHLSADEKSAINAVWGKVDIAGADGATLYAARLFLAFYDQKQRFAAFTADLSSQSAIGHKPKVHAHGKKI
"""
with open("hbb_alignment.fasta", "w") as f:
    f.write(alignment_content)

alignment = AlignIO.read("hbb_alignment.fasta", "fasta")
print(f"Sequences: {len(alignment)}, Length: {alignment.get_alignment_length()} columns\n")
for record in alignment:
    print(f"  {record.id:15s} {record.seq[:50]}...")

In [ ]:
# Alignment slicing and conservation
from collections import Counter

print(f"Column 0: {alignment[:, 0]}")
print(f"Columns 0-9:")
for rec in alignment[:, 0:10]:
    print(f"  {rec.id:15s} {rec.seq}")

# Calculate conservation
n_cols = alignment.get_alignment_length()
n_identical = 0
for i in range(n_cols):
    col = alignment[:, i]
    counts = Counter(col)
    counts.pop('-', None)
    if counts and counts.most_common(1)[0][1] == len(alignment):
        n_identical += 1

print(f"\nIdentical columns: {n_identical}/{n_cols} ({n_identical/n_cols*100:.1f}%)")

In [ ]:
# Pairwise alignment with Bio.Align
aligner = Align.PairwiseAligner()

# Global alignment (Needleman-Wunsch)
aligner.mode = "global"
alignments = aligner.align(Seq("MVHLTPEEKSAVTALWGKVN"), Seq("MVHLTDAEKAAVNGLWGKVN"))
print(f"Global alignment (score={alignments[0].score}):")
print(alignments[0])

# Local alignment (Smith-Waterman)
aligner.mode = "local"
alignments = aligner.align(Seq("XXXXMVHLTPEEKXXXXXX"), Seq("YYYMVHLTDAEKYYYY"))
print(f"Local alignment (score={alignments[0].score}):")
print(alignments[0])

---
## 6. Complete Workflow: Gene to Protein Analysis

Combining everything: fetch a gene from NCBI, extract CDS, translate, and analyze the protein.

In [ ]:
def gene_to_protein_analysis(gene_name, organism="Homo sapiens"):
    """
    Complete workflow: fetch gene from NCBI -> extract CDS -> translate -> analyze.
    """
    Entrez.email = "your.email@example.com"
    
    # Step 1: Search NCBI
    handle = Entrez.esearch(
        db="nucleotide",
        term=f"{gene_name}[Gene] AND {organism}[Organism] AND RefSeq[Filter] AND mRNA[Filter]",
        retmax=1
    )
    search_results = Entrez.read(handle)
    handle.close()
    
    if not search_results["IdList"]:
        print(f"No results for {gene_name}")
        return None
    
    # Step 2: Download GenBank record
    ncbi_id = search_results["IdList"][0]
    handle = Entrez.efetch(db="nucleotide", id=ncbi_id, rettype="gb", retmode="text")
    gb_record = SeqIO.read(handle, "genbank")
    handle.close()
    
    # Step 3: Extract CDS and translate
    cds_seq = None
    for feat in gb_record.features:
        if feat.type == "CDS":
            cds_seq = feat.location.extract(gb_record.seq)
            break
    
    if cds_seq is None:
        print(f"No CDS found for {gene_name}")
        return None
    
    protein = cds_seq.translate(to_stop=True)
    
    # Step 4: Analyze
    mw = molecular_weight(protein, seq_type="protein")
    gc = gc_fraction(gb_record.seq) * 100
    
    # Amino acid composition (top 5)
    aa_counts = {}
    for aa in str(protein):
        aa_counts[aa] = aa_counts.get(aa, 0) + 1
    
    result = {
        "gene": gene_name, "accession": gb_record.id,
        "mrna_bp": len(gb_record), "cds_bp": len(cds_seq),
        "protein_aa": len(protein), "mw_kda": mw / 1000,
        "gc_pct": gc,
    }
    
    print(f"{gene_name}: {gb_record.id} | mRNA {len(gb_record)} bp | "
          f"CDS {len(cds_seq)} bp | Protein {len(protein)} aa | "
          f"{mw/1000:.1f} kDa | GC {gc:.1f}%")
    
    # Save protein FASTA
    protein_record = SeqRecord(
        protein, id=f"{gene_name}_protein",
        description=f"{gene_name} protein from {gb_record.id}"
    )
    SeqIO.write([protein_record], f"{gene_name}_protein.fasta", "fasta")
    
    return result

In [ ]:
# Run for a single gene
result = gene_to_protein_analysis("INS")

In [ ]:
# Compare multiple genes
genes = ["HBB", "INS", "TP53"]
all_results = []

for gene in genes:
    r = gene_to_protein_analysis(gene)
    if r:
        all_results.append(r)

print(f"\n{'Gene':<8} {'mRNA':>8} {'CDS':>8} {'Protein':>10} {'MW(kDa)':>10} {'GC%':>8}")
print("-" * 56)
for r in all_results:
    print(f"{r['gene']:<8} {r['mrna_bp']:>8} {r['cds_bp']:>8} "
          f"{r['protein_aa']:>10} {r['mw_kda']:>10.1f} {r['gc_pct']:>8.1f}")

---
## 7. Useful Patterns

In [ ]:
# Pattern 1: Find all ORFs in a sequence
def find_orfs(seq, min_length=100):
    """Find ORFs in all 6 reading frames. min_length in nucleotides."""
    orfs = []
    for strand, nuc_seq in [('+', seq), ('-', seq.reverse_complement())]:
        s = str(nuc_seq)
        for frame in range(3):
            for i in range(frame, len(s) - 2, 3):
                if s[i:i+3] == 'ATG':
                    for j in range(i + 3, len(s) - 2, 3):
                        if s[j:j+3] in ('TAA', 'TAG', 'TGA'):
                            orf_len = j + 3 - i
                            if orf_len >= min_length:
                                orfs.append({
                                    'start': i, 'end': j+3, 'length': orf_len,
                                    'strand': strand, 'frame': frame+1,
                                    'protein': str(Seq(s[i:j+3]).translate(to_stop=True))
                                })
                            break
    return sorted(orfs, key=lambda x: -x['length'])

test_seq = Seq(
    "CCCATGGCCATTGTAATGGGCCGCTGAAAGCCCCCCCCCCCCCCC"
    "ATGTTTAAAGGGCCCAAATTTGGGCCCAAATTTGGGCCCAAATTTGGGCCCTAG"
)
for orf in find_orfs(test_seq, min_length=30):
    print(f"  Strand {orf['strand']} frame {orf['frame']}: {orf['length']} nt -> {orf['protein'][:30]}")

In [ ]:
# Pattern 2: Batch FASTA analysis
print(f"{'ID':<20} {'Length':>8} {'GC%':>8}")
print("-" * 38)
for rec in SeqIO.parse("insulin_seqs.fasta", "fasta"):
    print(f"{rec.id:<20} {len(rec):>8} {gc_fraction(rec.seq)*100:>7.1f}%")

# Pattern 3: Codon usage table
cds = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGA")
codons = {}
for i in range(0, len(cds) - 2, 3):
    c = str(cds[i:i+3])
    codons[c] = codons.get(c, 0) + 1

print(f"\nCodon usage for {cds}:")
for codon, count in sorted(codons.items()):
    print(f"  {codon} ({Seq(codon).translate()}): {count}")

---
## 8. Exercises

### Exercise 1: Sequence Manipulation

Given the DNA sequence below:
1. Find all occurrences of the restriction site `GAATTC` (EcoRI)
2. Calculate the GC content
3. Translate in all 3 forward reading frames
4. Find the longest ORF (starting with ATG, ending at a stop codon)

In [ ]:
exercise_seq = Seq(
    "ATGCGATCGAATTCGATCGATCGATCGATCGATCGATCG"
    "GAATTCATGGCCATTGTAATGGGCCGCTGAAAGGGTGCC"
    "CGATAGCCCGAATTCGCGATCGATCGATCGATCGATCGA"
)

# Exercise 1: Your code here


### Exercise 2: GenBank Analysis

Fetch the GenBank record for human hemoglobin alpha (HBA1), accession `NM_000558.5`. List all features, extract the CDS, translate it, and calculate the molecular weight. How many exons does it have?

In [ ]:
# Exercise 2: Your code here


### Exercise 3: FASTQ Quality Control

Write a function `fastq_qc(filename)` that reads a FASTQ file, reports total reads / average length / average quality, filters out reads with mean quality < 20, and writes passing reads to a new file. Test with `reads.fastq`.

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Comparative Analysis

Using `insulin_seqs.fasta`: perform pairwise global alignment for each pair of sequences, calculate percent identity, and report which species pair has the most similar insulin.

In [ ]:
# Exercise 4: Your code here


### Exercise 5: Gene Fetching Pipeline

Write a function that takes a list of gene names, fetches their human RefSeq mRNA from NCBI, extracts CDS, translates, and writes all proteins to a single FASTA file. Test with `["EGFR", "KRAS", "BRAF"]`.

In [ ]:
# Exercise 5: Your code here


---
## Cleanup

In [ ]:
import os

for f in ["insulin_seqs.fasta", "output.fasta", "example.gb", "reads.fastq",
          "filtered.fastq", "from_genbank.fasta", "reads.fasta",
          "ncbi_genes.fasta", "hbb_alignment.fasta",
          "INS_protein.fasta", "HBB_protein.fasta", "TP53_protein.fasta"]:
    if os.path.exists(f):
        os.remove(f)
print("Cleanup complete.")

---
## Summary

### Core Classes
- **`Seq`** -- biological sequence: `complement()`, `reverse_complement()`, `transcribe()`, `translate()`
- **`SeqRecord`** -- sequence + metadata (id, annotations, features, letter_annotations)
- **`SeqFeature`** -- annotated region on a sequence (gene, CDS, exon)

### Core Modules
- **`SeqIO`** -- read/write FASTA, GenBank, FASTQ, and many more
- **`AlignIO`** -- read/write alignment files
- **`Entrez`** -- NCBI database access (esearch, efetch, elink)
- **`Align`** -- pairwise alignment (global/local)
- **`SeqUtils`** -- GC content, molecular weight

### Quick Reference
| Operation | Code |
|-----------|------|
| Translate | `seq.translate(to_stop=True)` |
| Reverse complement | `seq.reverse_complement()` |
| Read file | `SeqIO.parse("file.fasta", "fasta")` |
| Write file | `SeqIO.write(records, "out.fasta", "fasta")` |
| Search NCBI | `Entrez.esearch(db="nucleotide", term="...")` |
| Fetch from NCBI | `Entrez.efetch(db="nucleotide", id="...", rettype="gb")` |

---

*Tier 2: Core Bioinformatics -- BioPython Essentials*

---[< Previous: Biological Databases](../01_Biological_Databases/01_biological_databases.ipynb) | [Tier 2: Core Bioinformatics Overview](../README.md) | [Next: Pairwise Sequence Alignment >](../03_Pairwise_Sequence_Alignment/01_pairwise_sequence_alignment.ipynb)---